# Transformación de la encuesta de Forms a los datasets `catalog` e `historical`

Este notebook toma las respuestas de la encuesta de Microsoft Forms guardadas en **Registro de procesos empresariales.xlsx** (OneDrive) y las convierte en los dos datasets del ejercicio:

| Dataset | Qué contiene | Regla |
|---|---|---|
| `catalog` | Procesos candidatos a automatizar | Respuestas donde **no** hay IA implementada |
| `historical` | Proyectos con resultado conocido | Respuestas donde **sí** hay IA implementada; `implementation_success` sale de la pregunta de éxito |

**Flujo**

1. Configuración 
2. Lectura del Excel
3. Identificación de las columnas
4. Limpieza y estandarización
5. Separación en `catalog` e `historical`
6. Ingeniería de características
7. Guardado

El notebook se puede volver a ejecutar completo cada vez que lleguen respuestas nuevas: genera los archivos desde cero a partir de todo el Excel.

### Correspondencia entre las preguntas y las variables

| Pregunta del formulario | Variable |
|---|---|
| ¿Qué departamento gestiona el proceso? | `department` |
| ¿Qué tipo de proceso es? | `process_name` |
| ¿Cuántas veces se ejecuta el proceso al mes? | `monthly_volume` |
| ¿Cuántos minutos toma cada ejecución? | `avg_minutes` |
| ¿Qué porcentaje del proceso se realiza manualmente? (0 a 100) | `manual_share` (0 a 1, se divide entre 100) |
| ¿Qué tan disponible y estructurada está la información necesaria? (1 a 5) | `data_availability` |
| ¿Qué tan complejo es el proceso? (1 a 5) | `process_complexity` |
| ¿Es un proceso con IA ya implementada? | decide si va a `historical` o `catalog` |
| ¿Fue la implementación con IA exitosa? | `implementation_success` (Sí = 1, No = 0) |

`Correo electrónico`, `Nombre` y las horas de inicio y finalización **no se llevan** a los datasets de salida.

`implementation_success` fue determinado en base al cumplimiento de dos criterios:
- El proyecto logró completarse con éxito, creando una herramienta funcional e integrada en los flujos de trabajo de la compañía.
- La herramienta generó un beneficio anual suficiente para recuperar la inversión en menos de 1 año.

## 1. Configuración

In [1]:
import requests
import io
import re
import html
import unicodedata
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

#! ---------------- Origen de los datos ----------------
ENLACE_ONEDRIVE = "https://1drv.ms/x/c/cef9065210287406/IQBrLHDGKua-TquWRVLIcjcYAVG4fPhth-kY9W7Jja5BBck?e=oIDn2w"  # enlace abierto a todo publico (no requiere autenticación)
HOJA = 0  # nombre o posición de la hoja con las respuestas

#! ---------------- Salida ----------------
CARPETA_SALIDA = Path("../data/outputs")
PREFIJO_CATALOG = "P"        # prefijo de los indices
PREFIJO_HISTORICAL = "H"     # prefijo de los indices

#! ---------------- Constantes ----------------
AVG_HOURLY_COST = 30000       # COP por hora (promedio de la compañía)

#! ---------------- Esquemas de salida ----------------
COLS_BASE = ["department", "process_name", "monthly_volume", "avg_minutes",
             "manual_share", "data_availability", "process_complexity"]
NUEVAS = ["annual_hours", "automation_potential", "recoverable_hours",
          "estimated_annual_benefit", "feasibility_score"]
COLS_CATALOG = ["process_id"] + COLS_BASE
COLS_HISTORICAL = ["project_id"] + COLS_BASE + ["implementation_success"]

## 2. Lectura del Excel

In [ ]:
# Definición de una función de lectura de Excel en OneDrive
def leer_enlace(url, hoja):
    """
    Lee un archivo Excel desde un enlace de OneDrive y devuelve un DataFrame de pandas.

    Args:
        url (string): url del archivo Excel en OneDrive
        hoja (string): nombre o índice de la hoja a leer

    Returns:
        DataFrame:  pandas Dataframe resultante
    """
    session = requests.Session()
    
    # Obtener la página de OneDrive
    pagina = session.get(url, timeout=60).text
    
    # Extraer URL directa de descarga
    url_descarga = re.search(
        r'"FileUrlNoAuth"\s*:\s*"([^"]+)"', pagina
    ).group(1)
    
    url_descarga = html.unescape(
        url_descarga.replace("\\u0026", "&").replace("\\/", "/")
    )
    
    # Descargar y leer Excel
    archivo = session.get(url_descarga, timeout=60)
    
    return pd.read_excel(
        io.BytesIO(archivo.content),
        sheet_name=hoja,
        engine="openpyxl"
    )

# Lectura de los datos de Excel
df_crudo = leer_enlace(ENLACE_ONEDRIVE, HOJA)

print(f"Respuestas leídas: {len(df_crudo)} | columnas: {df_crudo.shape[1]}")
display(df_crudo.head())

Respuestas leídas: 4 | columnas: 14


,Id,Hora de inicio,Hora de finalización,Correo electrónico,Nombre,¿Qué departamento gestiona el proceso?,¿Qué tipo de proceso es?,¿Cuántas veces se ejecuta el proceso al mes?\n,¿Cuántos minutos toma cada ejecución?\n,¿Qué porcentaje del proceso se realiza manualmente?\n,¿Qué tan disponible y estructurada está la información necesaria?\n,"¿Qué tan complejo es el proceso? ¿Cuántas personas, herramientas y subprocesos requiere?\n",¿Es un proceso con IA ya implementada?,¿Fue la implementación con IA exitosa?
0,1,2026-09-24 10:04:37,2026-09-24 10:06:29,anonymous,NaN,Human Resources,Invoice validation,78,50.0,60,3,2,No,NaN
1,2,2026-09-24 12:51:44,2026-09-24 12:54:18,anónimo,NaN,Logistics,Contract review,2,68.5,100,1,3,Si,No
2,3,2026-09-24 13:02:59,2026-09-24 13:03:31,anónimo,NaN,Finance,Inventory update,30,34.6,60,5,2,No,NaN
3,4,2026-09-24 13:03:56,2026-09-24 13:04:28,anónimo,NaN,Customer Service,Data consolidation,65,45.0,90,3,1,Si,Si


## 3. Identificación de las columnas

Las preguntas de Forms llegan como encabezados largos, con tildes y saltos de línea al final (`\n`). En lugar de depender del texto exacto, cada columna se identifica por **palabras clave** del encabezado normalizado (minúsculas, sin tildes ni espacios repetidos). Si una palabra clave no encuentra exactamente una columna, el notebook se detiene con un mensaje claro, por ejemplo si alguien renombra una pregunta en el formulario.

In [ ]:
# Definición de funciones para normalizar texto y mapear columnas del DataFrame de la encuesta
def normalizar_texto(valor):
    """
    Minúsculas, sin tildes ni saltos de línea, espacios simples.

    Args:
        valor (str): texto a normalizar.

    Returns:
        str: texto normalizado en minúsculas, sin tildes ni espacios repetidos.
    """
    if pd.isna(valor):
        return ""
    texto = unicodedata.normalize("NFKD", str(valor)).encode("ascii", "ignore").decode()
    return re.sub(r"\s+", " ", texto).strip().lower()


# variable de destino -> palabras que TODAS deben aparecer en el encabezado normalizado
PALABRAS_CLAVE = {
    "department":                ["departamento"],
    "process_name":              ["tipo de proceso"],
    "monthly_volume":            ["veces", "al mes"],
    "avg_minutes":               ["minutos"],
    "manual_share":              ["porcentaje", "manualmente"],
    "data_availability":         ["disponible"],
    "process_complexity":        ["complejo"],
    "ai_implemented":            ["ia", "ya implementada"],
    "implementation_success":    ["exitosa"],
}

# Función para mapear las columnas del DataFrame según las palabras clave definidas
def mapear_columnas(df):
    """
    Mapea las columnas del DataFrame de la encuesta según las palabras clave definidas.

    Args:
        df (pd.DataFrame): DataFrame de la encuesta con los encabezados originales.

    Raises:
        ValueError: Si no se pueden identificar de manera única las columnas de la encuesta.

    Returns:
        tuple: Mapa de columnas y nombre de la columna de ID.
    """
    normalizadas = {c: normalizar_texto(c) for c in df.columns}
    mapa, problemas = {}, []
    for destino, claves in PALABRAS_CLAVE.items():
        candidatas = [c for c, n in normalizadas.items() if all(k in n for k in claves)]
        if len(candidatas) == 1:
            mapa[candidatas[0]] = destino
        else:
            problemas.append(f"  - {destino}: {len(candidatas)} coincidencias {candidatas}")
    if problemas:
        raise ValueError("No se pudieron identificar las columnas de la encuesta:\n" + "\n".join(problemas))

    col_id = next((c for c, n in normalizadas.items() if n == "id"), None)
    return mapa, col_id

# Mapeo de columnas del DataFrame crudo y visualización del mapa de columnas
mapa_columnas, col_id = mapear_columnas(df_crudo)

tabla_mapa = pd.DataFrame({
    "encabezado en el Excel": [re.sub(r"\s+", " ", c).strip() for c in mapa_columnas],
    "variable": list(mapa_columnas.values()),
})
display(tabla_mapa)

,encabezado en el Excel,variable
0,¿Qué departamento gestiona el proceso?,department
1,¿Qué tipo de proceso es?,process_name
2,¿Cuántas veces se ejecuta el proceso al mes?,monthly_volume
3,¿Cuántos minutos toma cada ejecución?,avg_minutes
4,¿Qué porcentaje del proceso se realiza manualm...,manual_share
5,¿Qué tan disponible y estructurada está la inf...,data_availability
6,¿Qué tan complejo es el proceso? ¿Cuántas pers...,process_complexity
7,¿Es un proceso con IA ya implementada?,ai_implemented
8,¿Fue la implementación con IA exitosa?,implementation_success


## 4. Limpieza y estandarización

- Se renombran las columnas y se descartan las personales (correo, nombre) y las horas.
- Si/No: se convierten a 1/0.

In [ ]:
# Funciones y transformaciones para estandarizar los datos de la encuesta
def a_binario(serie):
    """
    Convierte una serie de valores "Si" o "No" a valores binarios (1 o 0).

    Args:
        serie (Series): Serie de pandas con valores "Si" o "No".

    Returns:
        Series: Serie de pandas con valores binarios (1 para "Si", 0 para "No").
    """
    equivalencias = {"Si": 1, "No": 0}
    return serie.map(lambda v: equivalencias.get(v, np.nan) if pd.notna(v) else np.nan)

# Copia estandarizada
datos = df_crudo.copy()
datos.rename(columns=mapa_columnas, inplace=True)
datos["ai_implemented"] = a_binario(datos["ai_implemented"])
datos["implementation_success"] = a_binario(datos["implementation_success"])

# Descarte de columnas no mapeadas
datos = datos[[col_id, *COLS_BASE, "ai_implemented", "implementation_success"]]

# manual_share se divide entre 100.
datos["manual_share"] = round(datos["manual_share"] / 100, 4)

display(datos.head())

,Id,department,process_name,monthly_volume,avg_minutes,manual_share,data_availability,process_complexity,ai_implemented,implementation_success
0,1,Human Resources,Invoice validation,78,50.0,0.6,3,2,0,NaN
1,2,Logistics,Contract review,2,68.5,1.0,1,3,1,0.0
2,3,Finance,Inventory update,30,34.6,0.6,5,2,0,NaN
3,4,Customer Service,Data consolidation,65,45.0,0.9,3,1,1,1.0


Se revisan tipos, nulos, duplicados, unicidad de identificadores y que los valores estén dentro de los rangos esperados.

In [ ]:
# Función para generar un resumen de la calidad de los datos de un DataFrame

def resumen_calidad(df, nombre, id_col):
    """
    Esta función imprime un resumen de la calidad de los datos de un DataFrame, incluyendo:
    - Número de filas y columnas
    - Número de filas duplicadas (excluyendo la columna de ID)
    - Unicidad de la columna de ID
    - Tabla con el tipo de dato, número de valores nulos, porcentaje de valores nulos y número de valores únicos por columna

    Args:
        df (pd.DataFrame): DataFrame a evaluar.
        nombre (str): Nombre descriptivo del DataFrame.
        id_col (str): Nombre de la columna de ID.
    """
    print(f"{nombre}: {df.shape[0]} filas x {df.shape[1]} columnas")
    print(f"  Filas duplicadas: {df.duplicated(subset=df.columns.difference([id_col])).sum()}")
    print(f"  {id_col} único: {df[id_col].is_unique}")
    tabla = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "nulos": df.isna().sum(),
        "% nulos": (df.isna().mean() * 100).round(2),
        "valores únicos": df.nunique(),
    })
    display(tabla)

# Resumen de la calidad de los datos del DataFrame 'datos'
resumen_calidad(datos, "datos", col_id)

datos: 4 filas x 10 columnas
  Filas duplicadas: 0
  Id único: True


,dtype,nulos,% nulos,valores únicos
Id,int64,0,0.0,4
department,object,0,0.0,4
process_name,object,0,0.0,4
monthly_volume,int64,0,0.0,4
avg_minutes,float64,0,0.0,4
manual_share,float64,0,0.0,3
data_availability,int64,0,0.0,3
process_complexity,int64,0,0.0,3
ai_implemented,int64,0,0.0,2
implementation_success,float64,2,50.0,2


## 5. Separación en `catalog` e `historical`

- `ai_implemented = 0` → **catalog** (`process_id`)
- `ai_implemented = 1` → **historical** (`project_id` y `implementation_success`)

Los identificadores se forman con el prefijo y el `Id` de Forms, así cada fila se puede rastrear hasta su respuesta original.

In [ ]:
# Texto del ID con ceros a la izquierda
id_texto = datos["Id"].astype(int).astype(str).str.zfill(4)

# Filas históricas y de catálogo
es_historical = datos["ai_implemented"] == 1

catalog = datos.loc[~es_historical].copy()
catalog["process_id"] = PREFIJO_CATALOG + id_texto[~es_historical]
catalog = catalog[COLS_CATALOG].sort_values("process_id").reset_index(drop=True)

historical = datos.loc[es_historical].copy()
historical["project_id"] = PREFIJO_HISTORICAL + id_texto[es_historical]
historical["implementation_success"] = historical["implementation_success"].astype(int)
historical = historical[COLS_HISTORICAL].sort_values("project_id").reset_index(drop=True)

print(f"catalog   : {len(catalog)} filas")
display(catalog.head())
print(f"historical: {len(historical)} filas")
display(historical.head())

catalog   : 2 filas


,process_id,department,process_name,monthly_volume,avg_minutes,manual_share,data_availability,process_complexity
0,P0001,Human Resources,Invoice validation,78,50.0,0.6,3,2
1,P0003,Finance,Inventory update,30,34.6,0.6,5,2


historical: 2 filas


,project_id,department,process_name,monthly_volume,avg_minutes,manual_share,data_availability,process_complexity,implementation_success
0,H0002,Logistics,Contract review,2,68.5,1.0,1,3,0
1,H0004,Customer Service,Data consolidation,65,45.0,0.9,3,1,1


## 6. Ingeniería de características

Se crean cinco variables nuevas con la misma función en ambos conjuntos, para que las fórmulas sean idénticas en entrenamiento y en predicción:

| Variable | Fórmula |
|---|---|
| `annual_hours` | `monthly_volume * avg_minutes * 12 / 60` |
| `automation_potential` | `0.50 * manual_share + 0.30 * (data_availability / 5) + 0.20 * (1 - process_complexity / 5)` |
| `recoverable_hours` | `annual_hours * automation_potential` |
| `estimated_annual_benefit` | `recoverable_hours * AVG_HOURLY_COST` (COP) |
| `feasibility_score` | `0.60 * (data_availability / 5) + 0.40 * (1 - process_complexity / 5)` |

`annual_hours` mide cuánto trabajo consume el proceso al año, `automation_potential` qué fracción es automatizable, y `feasibility_score` qué tan viable es técnicamente (datos disponibles y baja complejidad).

In [ ]:
# Función para agregar características derivadas a los DataFrames históricos y de catálogo
def add_features(df):
    """
    Agrega características derivadas a un DataFrame, incluyendo:
    - annual_hours: Horas anuales estimadas.
    - automation_potential: Potencial de automatización.
    - recoverable_hours: Horas recuperables.
    - estimated_annual_benefit: Beneficio anual estimado.
    - feasibility_score: Puntaje de factibilidad.

    Args:
        df (pd.DataFrame): DataFrame al que se agregarán las características.

    Returns:
        pd.DataFrame: DataFrame con las características derivadas agregadas.
    """
    df = df.copy()
    df["annual_hours"] = (df["monthly_volume"] * df["avg_minutes"] * 12 / 60).round(4)
    df["automation_potential"] = (
        0.50 * df["manual_share"]
        + 0.30 * (df["data_availability"] / 5)
        + 0.20 * (1 - df["process_complexity"] / 5)
    ).round(4)
    df["recoverable_hours"] = (df["annual_hours"] * df["automation_potential"]).round(2)
    df["estimated_annual_benefit"] = (df["recoverable_hours"] * AVG_HOURLY_COST).round(0)
    df["feasibility_score"] = (
        0.60 * (df["data_availability"] / 5)
        + 0.40 * (1 - df["process_complexity"] / 5)
    ).round(2)
    return df

# Agregar características derivadas a los DataFrames históricos y de catálogo
historical_fe = historical.copy()
historical_fe = add_features(historical_fe)

catalog_fe = catalog.copy()
catalog_fe = add_features(catalog_fe)

display(catalog_fe.head(3))
display(historical_fe.head(3))

,process_id,department,process_name,monthly_volume,avg_minutes,manual_share,data_availability,process_complexity,annual_hours,automation_potential,recoverable_hours,estimated_annual_benefit,feasibility_score
0,P0001,Human Resources,Invoice validation,78,50.0,0.6,3,2,780.0,0.60,468.00,14040000.0,0.60
1,P0003,Finance,Inventory update,30,34.6,0.6,5,2,207.6,0.72,149.47,4484100.0,0.84


,project_id,department,process_name,monthly_volume,avg_minutes,manual_share,data_availability,process_complexity,implementation_success,annual_hours,automation_potential,recoverable_hours,estimated_annual_benefit,feasibility_score
0,H0002,Logistics,Contract review,2,68.5,1.0,1,3,0,27.4,0.64,17.54,526200.0,0.28
1,H0004,Customer Service,Data consolidation,65,45.0,0.9,3,1,1,585.0,0.79,462.15,13864500.0,0.68


## 7. Guardado

In [ ]:
# Guardado de los DataFrames procesados en archivos CSV
CARPETA_SALIDA.mkdir(parents=True, exist_ok=True)
catalog_fe.to_csv(CARPETA_SALIDA / "1A_process_catalog_1.csv", index=False)
historical_fe.to_csv(CARPETA_SALIDA / "1A_historical_ai_projects_1.csv", index=False)

print(f"\nArchivos guardados en: {CARPETA_SALIDA.resolve()}")


Archivos guardados en: C:\Users\ascal\Documents\Pruebas Técnicas\prueba_tecnica-ServiceLab\data\outputs
